# 01 - Data Preparation

## Objective

The objective of this notebook is to transform the raw monthly student performance reports into a clean and consistent master dataset for analysis.

The preparation process includes:

- Inspecting worksheet structures
- Applying business rules
- Standardizing column names
- Creating a standardized IT assessment score
- Adding a reporting month column
- Standardizing the dataset structure
- Merging all monthly reports into one dataset
- Validating the prepared dataset
- Exporting the cleaned dataset

The output of this notebook will be used in all subsequent notebooks for exploratory data analysis, visualization, and dashboard development.

### Import Libraries

The following libraries are used for data preparation and transformation.

In [98]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

### Load Raw Dataset

Load the Excel workbook containing the monthly student performance reports.

Each worksheet represents a single month's student performance.

In [99]:
FILE_PATH = "../data/raw/Basic Course Score.xlsx"

excel = pd.ExcelFile(FILE_PATH)

datasets = {
    sheet: pd.read_excel(FILE_PATH, sheet_name=sheet)
    for sheet in excel.sheet_names
}

print(excel.sheet_names)

['March', 'April', 'May', 'June']


### Inspect Dataset Structure

Before performing any transformation, the structure of each worksheet is inspected to understand the available columns and identify structural differences between monthly reports.

This ensures that the data preparation process is based on the actual structure of the source data.

#### Summary

In [100]:
summary = []

for month, df in datasets.items():
    summary.append({
        "Month": month,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    })

summary_df = pd.DataFrame(summary)
print(summary_df)

   Month  Rows  Columns
0  March    56       17
1  April    55       18
2    May    54       17
3   June    53       15


#### Column Names

In [101]:
for month, df in datasets.items():
    print('=' * 30)
    print(month)
    print('=' * 30)

    for col in df.columns:
        print(col)

    print()

March
Rank
Name
Gender
Class
Java (35%)
Web (25%)
Korean (30%)
Att (10%)
Total (100%)
Scho.
Korean Benefit
Lunch
PA
M.S
E.A
Total
Comment

April
Rank
Name
Gender
Class
Java (35%)
Web (25%)
Korean (30%)
Att (10%)
Total (100%)
Scho.
Korean Benefit
Lunch
PA
M.S
Coding Challenge
E.A
Total
Comment

May
Rank
Name
Gender
Class
Project Topic
IT (60%)
Korean (30%)
Att (10%)
Total (100%)
Scho.
Korean Benefit
Lunch
PA
M.S
E.A
Total
Comment

June
Rank
Name
Gender
Class
Project Topic
IT (90%)
Att (10%)
Total (100%)
Scho.
Lunch
PA
M.S
E.A
Total
Comment



#### Data Types

In [102]:
for month, df in datasets.items():

    print(f"\n===== {month} =====")

    df.info()


===== March =====
<class 'pandas.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Rank            56 non-null     int64  
 1   Name            56 non-null     str    
 2   Gender          56 non-null     str    
 3   Class           56 non-null     str    
 4   Java (35%)      56 non-null     float64
 5   Web (25%)       56 non-null     float64
 6   Korean (30%)    56 non-null     float64
 7   Att (10%)       56 non-null     float64
 8   Total (100%)    56 non-null     float64
 9   Scho.           56 non-null     int64  
 10  Korean Benefit  56 non-null     int64  
 11  Lunch           56 non-null     int64  
 12  PA              56 non-null     int64  
 13  M.S             4 non-null      float64
 14  E.A             3 non-null      float64
 15  Total           56 non-null     float64
 16  Comment         13 non-null     str    
dtypes: float64(8), int64(5), str(

### Understand Assessment Structure

The assessment criteria changed during the reporting period.

##### March – April

| Assessment | Weight |
|------------|--------|
| Java | 35% |
| Web | 25% |
| Korean | 30% |
| Attendance | 10% |

##### May

| Assessment | Weight |
|------------|--------|
| IT | 60% |
| Korean | 30% |
| Attendance | 10% |

##### June

| Assessment | Weight |
|------------|--------|
| IT | 90% |
| Attendance | 10% |

##### Business Rule

Java and Web represent the IT assessment during March and April.

To create a consistent dataset for longitudinal analysis, the IT score for March and April will be calculated as:

**IT = Java + Web**

The existing IT scores for May and June will be retained.

### Standardize Column Names

The monthly reports use different column names and abbreviations to represent the same information.

To create a consistent master dataset, column names are standardized while preserving the original meaning of the data.

In [103]:
COLUMN_MAPPING = {
    "Name": "Student Name",
    "Att (10%)": "Attendance",
    "Scho.": "Scholarship",
    "M.S": "Meal Support",
    "E.A": "Extra Activity",
    "IT (60%)": "IT",
    "IT (90%)": "IT",
    "Java (35%)": "Java",
    "Web (25%)": "Web",
    "Korean (30%)": "Korean",
    "Total": "Total IT Score",
    "Total (100%)": "Total",
}


def standardize_column_names(df):
    """
    Rename columns using the predefined column mapping.
    """
    return df.rename(columns=COLUMN_MAPPING)

In [104]:
for month in datasets:
    datasets[month] = standardize_column_names(datasets[month])

In [105]:
for month, df in datasets.items():
    print("=" * 80)
    print(month)
    print("=" * 80)

    print(df.columns.tolist)
    print()

March
<bound method IndexOpsMixin.tolist of Index(['Rank', 'Student Name', 'Gender', 'Class', 'Java', 'Web', 'Korean', 'Attendance', 'Total', 'Scholarship', 'Korean Benefit', 'Lunch', 'PA', 'Meal Support', 'Extra Activity', 'Total IT Score',
       'Comment'],
      dtype='str')>

April
<bound method IndexOpsMixin.tolist of Index(['Rank', 'Student Name', 'Gender', 'Class', 'Java', 'Web', 'Korean', 'Attendance', 'Total', 'Scholarship', 'Korean Benefit', 'Lunch', 'PA', 'Meal Support', 'Coding Challenge', 'Extra Activity',
       'Total IT Score', 'Comment'],
      dtype='str')>

May
<bound method IndexOpsMixin.tolist of Index(['Rank', 'Student Name', 'Gender', 'Class', 'Project Topic', 'IT', 'Korean', 'Attendance', 'Total', 'Scholarship', 'Korean Benefit', 'Lunch', 'PA', 'Meal Support', 'Extra Activity',
       'Total IT Score', 'Comment'],
      dtype='str')>

June
<bound method IndexOpsMixin.tolist of Index(['Rank', 'Student Name', 'Gender', 'Class', 'Project Topic', 'IT', 'Attendance'

### Create Standardized IT Score

The assessment structure changed during the reporting period.

For March and April, the IT assessment is represented by separate Java and Web scores.

A standardized **IT** column is created by combining the Java and Web scores.

For May and June, the existing IT scores are retained.

In [106]:
def create_it_score(df):
    """
    Creates a standardized IT score.

    For March and April:
        IT = Java + Web

    For May and June:
        Retains the existing IT score.

    Parameters
    ----------
    df : pandas.DataFrame

    Returns
    -------
    pandas.DataFrame
    """

    if {"Java", "Web"}.issubset(df.columns):
        df["IT"] = df["Java"] + df["Web"]

    return df

In [107]:
for month in datasets:
    datasets[month] = create_it_score(datasets[month])

In [108]:
for month, df in datasets.items():
    print("=" * 80)
    print(month)
    print("=" * 80)

    if "IT" in df.columns:
        print(df[["Student Name", "IT"]].head())

March
        Student Name     IT
0        CHHIM POJIM  46.49
1            TY DINE  44.33
2       DONG SIENGLY  44.96
3       UN SOVANNARA  42.14
4  SOEUN SOVANNARITH  41.80
April
     Student Name     IT
0    DONG SIENGLY  45.34
1         TY DINE  40.75
2      HEAV SEIΜΑ  43.73
3      CHHIM POЛМ  40.72
4  ROERN CHAMREUN  47.40
May
   Student Name     IT
0  DONG SIENGLY  48.56
1       TY DINE  49.28
2    CHHIM POJM  51.11
3     TIP DALIN  49.87
4    HEAV SEIMA  49.81
June
          Student Name     IT
0    SOEUN SOVANNARITH  81.81
1       PHENG MENGHEAK  81.56
2     CHEAM NORAKPANHA  81.06
3  SOVANNOEUT SREYNEAT  80.31
4        KEO VUTHTHANA  79.96


## Progress Check

At this stage, each worksheet has been prepared individually.

The following transformations have been completed:

- Column names standardized.
- IT score standardized.
- Month column added.

The next step is to align all worksheets to a common structure before merging them into a single master dataset.

### Add Month Column

Each worksheet represents a different reporting month.

To preserve this information after merging the worksheets, a new **Month** column is added to every dataset.

This column enables month-to-month comparisons during the exploratory data analysis phase.

In [109]:
def add_month_column(df, month):
    """
    Add a reporting month to each record.
    """

    df["Month"] = month
    return df

In [110]:
for month, df in datasets.items():
    datasets[month] = add_month_column(df, month)

In [111]:
for month, df in datasets.items():
    print("=" * 70)
    print(month)
    print("=" * 70)

    display(df[["Month", "Student Name"]].head())

March


,Month,Student Name
0,March,CHHIM POJIM
1,March,TY DINE
2,March,DONG SIENGLY
3,March,UN SOVANNARA
4,March,SOEUN SOVANNARITH


April


,Month,Student Name
0,April,DONG SIENGLY
1,April,TY DINE
2,April,HEAV SEIΜΑ
3,April,CHHIM POЛМ
4,April,ROERN CHAMREUN


May


,Month,Student Name
0,May,DONG SIENGLY
1,May,TY DINE
2,May,CHHIM POJM
3,May,TIP DALIN
4,May,HEAV SEIMA


June


,Month,Student Name
0,June,SOEUN SOVANNARITH
1,June,PHENG MENGHEAK
2,June,CHEAM NORAKPANHA
3,June,SOVANNOEUT SREYNEAT
4,June,KEO VUTHTHANA


### Standardize Dataset Structure

The monthly reports contain different assessment components due to curriculum changes.

Before merging the worksheets into a single dataset, all worksheets are aligned to a common structure.

Columns that are not applicable to a particular month are retained with missing values (`NaN`). This preserves the original meaning of the data while ensuring a consistent dataset structure for analysis.

#### Define the Master Schema

In [ ]:
MASTER_COLUMNS = [
    "Month",
    "Rank",
    "Student Name",
    "Gender",
    "Class",
    "Project Topic",
    "Coding Challenge",
    "Java",
    "Web",
    "IT",
    "Total IT Score",
    "Korean",
    "Attendance",
    "Scholarship",
    "Meal Support",
    "Lunch",
    "PA",
    "Extra Activity",
    "Comment",
    "Total",
]

#### Align Every Worksheet

In [114]:
def standardize_dataset_structure(df):
    """
    Align the dataset to the predefined master schema.
    """
    return df.reindex(columns=MASTER_COLUMNS)

In [115]:
for month in datasets:
    datasets[month] = standardize_dataset_structure(datasets[month])

#### Verify

In [116]:
for month, df in datasets.items():
    print("=" * 70)
    print(month)
    print("=" * 70)

    print(df.columns.tolist())

March
['Month', 'Rank', 'Student Name', 'Gender', 'Class', 'Project Topic', 'Coding Challenge', 'Java', 'Web', 'IT', 'IT Total', 'Korean', 'Attendance', 'Scholarship', 'Meal Support', 'Lunch', 'PA', 'Extra Activity', 'Comment', 'Total']
April
['Month', 'Rank', 'Student Name', 'Gender', 'Class', 'Project Topic', 'Coding Challenge', 'Java', 'Web', 'IT', 'IT Total', 'Korean', 'Attendance', 'Scholarship', 'Meal Support', 'Lunch', 'PA', 'Extra Activity', 'Comment', 'Total']
May
['Month', 'Rank', 'Student Name', 'Gender', 'Class', 'Project Topic', 'Coding Challenge', 'Java', 'Web', 'IT', 'IT Total', 'Korean', 'Attendance', 'Scholarship', 'Meal Support', 'Lunch', 'PA', 'Extra Activity', 'Comment', 'Total']
June
['Month', 'Rank', 'Student Name', 'Gender', 'Class', 'Project Topic', 'Coding Challenge', 'Java', 'Web', 'IT', 'IT Total', 'Korean', 'Attendance', 'Scholarship', 'Meal Support', 'Lunch', 'PA', 'Extra Activity', 'Comment', 'Total']


### Merge Monthly Reports

After aligning the structure of each worksheet, the monthly datasets are merged into a single master dataset.

This master dataset will be used for all subsequent exploratory data analysis and visualization tasks.

In [117]:
master_df = pd.concat(
    datasets.values(),
    ignore_index=True
)

master_df.head()

,Month,Rank,Student Name,Gender,Class,Project Topic,Coding Challenge,Java,Web,IT,IT Total,Korean,Attendance,Scholarship,Meal Support,Lunch,PA,Extra Activity,Comment,Total
0,March,1,CHHIM POJIM,MALE,PP,NaN,NaN,28.32,18.17,46.49,NaN,25.32,10.0,45,NaN,30,10,NaN,CL,81.81
1,March,2,TY DINE,MALE,PVH,NaN,NaN,26.82,17.51,44.33,NaN,27.27,10.0,45,NaN,30,10,20.0,NaN,81.60
2,March,3,DONG SIENGLY,MALE,PP,NaN,NaN,28.25,16.71,44.96,NaN,26.31,10.0,45,1.0,30,10,NaN,NaN,81.27
3,March,4,UN SOVANNARA,MALE,PP,NaN,NaN,25.75,16.39,42.14,NaN,26.17,10.0,40,NaN,30,10,NaN,NaN,78.31
4,March,5,SOEUN SOVANNARITH,MALE,PVH,NaN,NaN,25.55,16.25,41.80,NaN,25.80,10.0,40,NaN,30,10,NaN,NaN,77.60


#### Verify

In [118]:
print(f"Dataset Shape: {master_df.shape}")

Dataset Shape: (218, 20)


In [119]:
print("Master Dataset Summary")
print("-" * 40)

print(f"Rows    : {master_df.shape[0]}")
print(f"Columns : {master_df.shape[1]}")

print("\nColumns:")

for col in master_df.columns:
    print(f"- {col}")

Master Dataset Summary
----------------------------------------
Rows    : 218
Columns : 20

Columns:
- Month
- Rank
- Student Name
- Gender
- Class
- Project Topic
- Coding Challenge
- Java
- Web
- IT
- IT Total
- Korean
- Attendance
- Scholarship
- Meal Support
- Lunch
- PA
- Extra Activity
- Comment
- Total
